# 2. Transición y Discretización (MountainCar)

En este nb damos el salto desde los entornos "Toy Text" a **Classic Control**, enfrentándonos a espacios de observación **continuos**.

Empezaremos con `MountainCar-v0`, un entorno donde un coche situado en un valle debe alcanzar la cima de la colina de la derecha, pero su motor no es lo suficientemente fuerte como para subir directamente; ¡debe balancearse acumulando inercia!

El estado devuelto por Gymnasium son números reales que representan `[posición, velocidad]`. **Dado que Q-Learning y SARSA utilizan tablas discretas, necesitamos "discretizar" este estado infinito.**

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# Componentes previamente desarrollados
from src.agents.agent import Agent
from src.policies.epsilon_greedy import EpsilonGreedyPolicy
from src.learners.q_learning import QLearning
from src.plotting.plotting import plot_rewards, plot_episode_lengths

# ¡Nuevo! La herramienta de discretización
from src.utils.discretizer import StateDiscretizer

## Discretización y Configuración

El truco de la discretización (*Binning*) consiste en dividir la posición y la velocidad en pequeñas "cajas". Cuantas más cajas, mayor precisión, pero más gigantesca se volverá la Q-Table (la maldición de la dimensionalidad).

In [ ]:
env_name = "MountainCar-v0"
env = gym.make(env_name)

# La acción sigue siendo discreta (0:Izq, 1:Nada, 2:Der)
action_size = env.action_space.n

# Discretizamos el espacio continuo (usamos 20 particiones o cajas por cada entrada)
# Posición continua y Velocidad continua se vuelven 20x20 = 400 estados
bins = (20, 20)
discretizer = StateDiscretizer(env, bins)
state_size = discretizer.get_state_size()

print(f"Límites continuos (posición): [{env.observation_space.low[0]}, {env.observation_space.high[0]}]")
print(f"Límites continuos (velocidad): [{env.observation_space.low[1]}, {env.observation_space.high[1]}]")
print(f"Tamaño de la tabla tras la discretización: {state_size} estados x {action_size} acciones.")

## Wrapper del Entorno para encajar en el Agente actual
Dado que nuestro `Agent` está diseñado para recibir un $s_t$ como número entero indexable, tenemos que interponer el Discretizer cada vez que Gymnasium suelte un estado nuevo continuo:

In [ ]:
class DiscretizedEnvWrapper(gym.ObservationWrapper):
    def __init__(self, env, discretizer):
        super().__init__(env)
        self.discretizer = discretizer
        self.observation_space = gym.spaces.Discrete(discretizer.get_state_size())

    def observation(self, observation):
        # Transforma el np.array continuo en un índice escalar entero (int)
        return self.discretizer.discretize(observation)

# Aplicar el wrapper
discrete_env = DiscretizedEnvWrapper(env, discretizer)

## Entrenamiento y Evaluación Tabular
Usaremos Q-Learning ahora que el entorno "finge" ser puramente discreto.

In [ ]:
n_episodes = 5000 # Requiere más episodios al sufrir la penalización tabular
n_runs = 5
gamma = 0.99
alpha = 0.1

policy = EpsilonGreedyPolicy(epsilon_start=1.0, epsilon_min=0.01, epsilon_decay=0.999)
q_learning = QLearning(state_size, action_size, alpha, gamma)

print("Entrenando Q-Learning Discretizado en MountainCar...")
agent = Agent(discrete_env, q_learning, policy)
qtable, rewards, lengths, stats = agent.train(num_episodes=n_episodes, n_runs=n_runs)

## Gráficas

Las gráficas deberían reflejar que, a pesar usar matrices, el aprendizaje acaba estabilizándose, aunque a gran costo (tanto de memoria a escala, como de episodios requeridos). Este cuello de botella será nuestra excusa matemática para la **Fase 4: Control con Aproximaciones**.

In [ ]:
plot_rewards([rewards], legend_labels=["Q-Learning (20x20 Bins)"], log_scale=False)
plot_episode_lengths([lengths], legend_labels=["Q-Learning (20x20 Bins)"], log_scale=False)